## Prediction 
We are going to predict from the model that we created.

In [1]:
import boto3

client = boto3.client("sagemaker-runtime")


response = client.invoke_endpoint(
    EndpointName='churnendpoint',
    Body="41,25.25,996.45,0,1,1,0,1,0,1,0,0,1,0,0,1,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,1,0,0,0,1,1,0,0,0",
    # Body="1,24.8,24.8,1,0,1,0,0,1,1,0,1,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0",
    ContentType='text/csv;label_size=1',
    Accept='text/csv',
)

for i in response.get("Body"):
    print(float(i))

0.04608782008290291


#### Code Explanation:
**Setting Up the SageMaker Runtime Client:**
- `boto3.client("sagemaker-runtime")` creates a client to communicate with deployed SageMaker endpoints.
- `invoke_endpoint()` sends a prediction request to our deployed `churnendpoint`:
  - **`Body`** — A comma-separated string of 46 feature values representing one customer's data.
  - **`ContentType="text/csv;label_size=1"`** — Tells the endpoint the input is CSV format, and the first value is the label (skipped during inference).
  - **`Accept="text/csv"`** — Requests the prediction output in CSV format.
- The response body is streamed back and we print the churn **probability score** (0 to 1). A score ≥ 0.5 indicates the customer is likely to churn.

In [2]:
# ============================================================
# GRADIO UI FOR CUSTOMER CHURN PREDICTION
# ============================================================

!pip install gradio -q

import gradio as gr
import boto3

client = boto3.client("sagemaker-runtime")
ENDPOINT_NAME = "churnendpoint"  # ← change if your endpoint name is different

def predict_churn(
    tenure, monthly_charges, total_charges,
    gender, senior_citizen, partner, dependents,
    phone_service, multiple_lines, internet_service,
    online_security, online_backup, device_protection,
    tech_support, streaming_tv, streaming_movies,
    contract, paperless_billing, payment_method
):
    # One-hot encode all inputs to match the 46-feature vector
    def ohe(value, options):
        return [1 if value == o else 0 for o in options]

    features = [
        tenure, monthly_charges, total_charges,
        *ohe(gender,           ["Female", "Male"]),
        *ohe(partner,          ["No", "Yes"]),
        *ohe(dependents,       ["No", "Yes"]),
        *ohe(phone_service,    ["No", "Yes"]),
        *ohe(multiple_lines,   ["No", "No phone service", "Yes"]),
        *ohe(internet_service, ["DSL", "Fiber optic", "No"]),
        *ohe(online_security,  ["No", "No internet service", "Yes"]),
        *ohe(online_backup,    ["No", "No internet service", "Yes"]),
        *ohe(device_protection,["No", "No internet service", "Yes"]),
        *ohe(tech_support,     ["No", "No internet service", "Yes"]),
        *ohe(streaming_tv,     ["No", "No internet service", "Yes"]),
        *ohe(streaming_movies, ["No", "No internet service", "Yes"]),
        *ohe(contract,         ["Month-to-month", "One year", "Two year"]),
        *ohe(paperless_billing,["No", "Yes"]),
        *ohe(payment_method,   [
            "Bank transfer (automatic)",
            "Credit card (automatic)",
            "Electronic check",
            "Mailed check"
        ]),
        *ohe(str(senior_citizen), ["0", "1"]),
    ]

    body = ",".join(str(f) for f in features)

    response = client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        Body=body,
        ContentType="text/csv;label_size=1",
        Accept="text/csv",
    )

    score = float(b"".join(response["Body"]))
    label = "🔴 Likely to Churn" if score >= 0.5 else "🟢 Likely to Stay"
    return f"{label}  (Churn probability: {score:.2%})"


# Build the UI
with gr.Blocks(title="Customer Churn Predictor") as demo:
    gr.Markdown("# 🔮 Customer Churn Predictor\nPowered by XGBoost on Amazon SageMaker")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Account Info")
            tenure          = gr.Slider(0, 72, value=12, step=1, label="Tenure (months)")
            monthly_charges = gr.Number(value=65.0, label="Monthly Charges ($)")
            total_charges   = gr.Number(value=780.0, label="Total Charges ($)")
            senior_citizen  = gr.Radio(["0", "1"], value="0", label="Senior Citizen")
            partner         = gr.Radio(["Yes", "No"], value="No", label="Partner")
            dependents      = gr.Radio(["Yes", "No"], value="No", label="Dependents")
            gender          = gr.Radio(["Female", "Male"], value="Female", label="Gender")

        with gr.Column():
            gr.Markdown("### 📞 Services")
            phone_service    = gr.Radio(["Yes", "No"], value="Yes", label="Phone Service")
            multiple_lines   = gr.Radio(["Yes", "No", "No phone service"], value="No", label="Multiple Lines")
            internet_service = gr.Radio(["DSL", "Fiber optic", "No"], value="DSL", label="Internet Service")
            online_security  = gr.Radio(["Yes", "No", "No internet service"], value="No", label="Online Security")
            online_backup    = gr.Radio(["Yes", "No", "No internet service"], value="No", label="Online Backup")
            device_protection= gr.Radio(["Yes", "No", "No internet service"], value="No", label="Device Protection")
            tech_support     = gr.Radio(["Yes", "No", "No internet service"], value="No", label="Tech Support")
            streaming_tv     = gr.Radio(["Yes", "No", "No internet service"], value="No", label="Streaming TV")
            streaming_movies = gr.Radio(["Yes", "No", "No internet service"], value="No", label="Streaming Movies")

        with gr.Column():
            gr.Markdown("### 💳 Billing")
            contract         = gr.Radio(["Month-to-month", "One year", "Two year"], value="Month-to-month", label="Contract")
            paperless_billing= gr.Radio(["Yes", "No"], value="Yes", label="Paperless Billing")
            payment_method   = gr.Dropdown([
                "Electronic check", "Mailed check",
                "Bank transfer (automatic)", "Credit card (automatic)"
            ], value="Electronic check", label="Payment Method")

            gr.Markdown("### 🎯 Prediction")
            predict_btn = gr.Button("Predict Churn", variant="primary")
            result      = gr.Textbox(label="Result", interactive=False)

    predict_btn.click(
        fn=predict_churn,
        inputs=[
            tenure, monthly_charges, total_charges,
            gender, senior_citizen, partner, dependents,
            phone_service, multiple_lines, internet_service,
            online_security, online_backup, device_protection,
            tech_support, streaming_tv, streaming_movies,
            contract, paperless_billing, payment_method
        ],
        outputs=result
    )

demo.launch(share=True)  # share=True gives a public URL

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://5b26fa4984826ad1fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#### Code Explanation:
**Gradio UI for Interactive Churn Prediction:**

This cell builds a fully interactive web application on top of the SageMaker endpoint.

**Key Components:**
- **`predict_churn()` function** — Takes all 19 customer attributes as inputs, one-hot encodes them into the 46-feature vector the model expects, calls the endpoint, and returns a human-readable result.
- **`ohe()` helper** — Converts a categorical value (e.g., `"DSL"`) into a binary list (e.g., `[1, 0, 0]`) matching the exact order used during training.
- **`gr.Blocks()`** — Builds the UI layout with 3 columns: Account Info, Services, and Billing.
- **`demo.launch(share=True)`** — Starts the app and generates a **public URL** valid for 1 week, so you can share or demo it without any additional hosting.

**Result Interpretation:**
- 🟢 **Likely to Stay** — Churn probability < 50%
- 🔴 **Likely to Churn** — Churn probability ≥ 50%